# Práctica 2: Soluciones de los ejercicios propuestos
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

Los ejercicios que se plantean a continuación tienen como objetivo el practicar con la biblioteca [Keras](https://keras.io/api/) de Python.

In [146]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

### Ejercicio 1

El hormigón es el material más importante en la ingeniería civil. La resistencia a la compresión del hormigón es una función altamente no lineal de su edad y sus ingredientes.

El fichero `concrete_data.csv` contiene la siguiente información acerca de diferentes muestras de hormigón:

* Contenido de cemento (`Cement`).
* Contenido de escoria de alto horno (`Blast Furnace Slag`).
* Contenido de cenizas volantes (`Fly Ash`).
* Contenido de agua (`Water`).
* Contenido de superplastificantes (`Superplasticizer`).
* Contenido de agregados gruesos (`Coarse Aggregate`).
* Contenido de agregados finos (`Fine Aggregate`).
* Edad del hormigón (`Age`).

El objetivo es predecir la resistencia a la compresión (`Strength`) a partir de esos atributos continuos.

Se pide realizar lo siguiente:

1. Dividir el conjunto de datos en un subconjunto de entrenamiento y un subconjunto de prueba.
2. Abordar la tarea mediante una red neuronal que tenga una única capa oculta con 16 neuronas y función de activación sigmoide.
3. Entrenar la red durante 50 épocas, usando el error cuadrático medio como función de coste a minimizar.
4. Calcular el rendimiento de la red como el error absoluto medio sobre el conjunto de prueba.
5. Experimentar con distintas arquitecturas de la red (cantidad de capas ocultas, cantidad de neuronas en cada capa oculta, función de activación de las capas) e hiperparámetros de entrenamiento (factor de aprendizaje, tamaño de los minilotes, número de épocas), tratando de encontrar una red con un error absoluto medio sobre el conjunto de prueba menor a 0.1.

In [147]:
import os
#os.environ["KERAS_BACKEND"] = "torch"

from keras.utils import set_random_seed

set_random_seed(982375)

In [148]:
import pandas as pd

In [149]:
hormigón = pd.read_csv('concrete_data.csv')
hormigón.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [150]:
hormigón.shape

(1030, 9)

In [151]:
hormigón.isna().any()

Cement                False
Blast Furnace Slag    False
Fly Ash               False
Water                 False
Superplasticizer      False
Coarse Aggregate      False
Fine Aggregate        False
Age                   False
Strength              False
dtype: bool

In [152]:
atributos = hormigón.loc[:, 'Cement':'Age']
atributos.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360


In [153]:
objetivo = hormigón.loc[:, ['Strength']]
objetivo.head()

,Strength
0,79.99
1,61.89
2,40.27
3,41.05
4,44.30


In [154]:
from sklearn.model_selection import train_test_split

In [155]:
(atributos_entrenamiento, atributos_prueba,
 objetivo_entrenamiento, objetivo_prueba) = train_test_split(
    atributos, objetivo,
    test_size=.2
)

In [156]:
from keras import Sequential, Input
from keras.layers import Dense, Normalization

In [157]:
normalizador = Normalization()
normalizador.adapt(atributos_entrenamiento.to_numpy())

Tomado de https://community.deeplearning.ai/t/doubt-about-relu-activation-in-hidden-layer/264792.


![Alt text](https://global.discourse-cdn.com/dlai/original/3X/d/4/d410f436378cd5af1ded43d0f4209bc3a5437e49.jpeg)

In [ ]:
red_hormigón = Sequential()
red_hormigón.add(Input(shape=(8,)))
red_hormigón.add(normalizador)
red_hormigón.add(Dense(16, activation='sigmoid'))
red_hormigón.add(Dense(1))

In [159]:
red_hormigón.compile(optimizer='SGD', loss='mean_squared_error',
                     metrics=['mean_absolute_error'])

In [160]:
red_hormigón.fit(atributos_entrenamiento, objetivo_entrenamiento,
                 batch_size=256, epochs=700)

Epoch 1/700
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1333.4763 - mean_absolute_error: 32.2738  
Epoch 2/700
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 653.8478 - mean_absolute_error: 20.5382 
Epoch 3/700
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 301.4130 - mean_absolute_error: 13.5175 
Epoch 4/700
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 191.0023 - mean_absolute_error: 10.7924 
Epoch 5/700
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 151.9274 - mean_absolute_error: 9.6875  
Epoch 6/700
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 131.4752 - mean_absolute_error: 9.0482 
Epoch 7/700
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 119.2835 - mean_absolute_error: 8.6168 
Epoch 8/700
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 111.5251 - mean_absolute_error: 8.3127 
Epoch 9/700
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 106.2366 - mean_absolute_error: 8.0804 
Epoch 10/700
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 102.3770 - mean_absolute_error: 7.8941 
Epoch 11/700
4/4 ━━━━━

In [161]:
red_hormigón.evaluate(atributos_prueba, objetivo_prueba)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28.2067 - mean_absolute_error: 4.1458 


[28.20671844482422, 4.1457600593566895]

In [162]:
from keras.optimizers import SGD

Funciones de activación comunes en las capas ocultas de las redes neuronales:

![Funciones de activacion](https://gkadusumilli.github.io/images/relu/activation_list.png)

In [163]:
red_hormigón = Sequential()
red_hormigón.add(Input(shape=(8,)))
red_hormigón.add(normalizador)
red_hormigón.add(Dense(4, activation='relu'))
red_hormigón.add(Dense(8, activation='relu'))
red_hormigón.add(Dense(1))

In [164]:
red_hormigón.compile(optimizer=SGD(learning_rate=0.0001), loss='mean_squared_error',
                     metrics=['mean_absolute_error'])

In [165]:
red_hormigón.fit(atributos_entrenamiento, objetivo_entrenamiento,
                 batch_size=64, epochs=300, verbose=False)

In [166]:
red_hormigón.evaluate(atributos_prueba, objetivo_prueba)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 96.0546 - mean_absolute_error: 8.0797  


[96.05463409423828, 8.079695701599121]

### Ejercicio 2

El fichero `cars.csv` contiene información acerca de la idoneidad de una serie de coches, en función de los siguientes atributos discretos:

* Precio de compra (`buying`): posibles valores `vhigh`, `high`, `med`, `low`.
* Coste de mantenimiento (`maint`): posibles valores `vhigh`, `high`, `med`, `low`.
* Número de puertas (`doors`): posibles valores `2`, `3`, `4`, `5more`.
* Número de asientos (`persons`): posibles valores `2`, `4`, `more`.
* Tamaño del maletero (`lug_boot`): posibles valores `small`, `med`, `big`.
* Nivel de seguridad estimada (`safety`): posibles valores `low`, `med`, `high`.

La idoneidad de cada coche se indica mediante el atributo `acceptability`, que los clasifica como `unacc`, `acc`, `good` o `vgood`.

Se pide realizar lo siguiente:

1. Dividir el conjunto de datos en un subconjunto de entrenamiento y un subconjunto de prueba.
2. Abordar la tarea mediante una red neuronal que tenga una única capa oculta con 16 neuronas y función de activación tangente hiperbólica.
3. Entrenar la red durante 50 épocas, usando la entropía cruzada categórica como función de coste a minimizar.
4. Calcular el rendimiento de la red como la tasa de acierto sobre el conjunto de prueba.
5. Experimentar con distintas arquitecturas de la red (cantidad de capas ocultas, cantidad de neuronas en cada capa oculta, función de activación de las capas) e hiperparámetros de entrenamiento (factor de aprendizaje, tamaño de los minilotes), tratando de encontrar una red que, entrenada únicamente durante 50 épocas, tenga una tasa de acierto sobre el conjunto de prueba superior a 0.9.

In [167]:
from keras.utils import set_random_seed

set_random_seed(3752385)

In [168]:
import pandas as pd

In [169]:
coches = pd.read_csv('cars.csv')
coches.head()

,buying,maint,doors,persons,lug_boot,safety,acceptability
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [170]:
coches.isna().any()

buying           False
maint            False
doors            False
persons          False
lug_boot         False
safety           False
acceptability    False
dtype: bool

In [171]:
from sklearn.preprocessing import OneHotEncoder, LabelBinarizer

In [172]:
codificador_atributos = OneHotEncoder(sparse_output=False)
atributos = coches.loc[:, 'buying':'safety']
atributos = codificador_atributos.fit_transform(atributos)
atributos

array([[0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 1., 0., 0.],
       ...,
       [0., 1., 0., ..., 0., 1., 0.],
       [0., 1., 0., ..., 0., 0., 1.],
       [0., 1., 0., ..., 1., 0., 0.]])

In [173]:
atributos.shape

(1728, 21)

In [174]:
codificador_objetivo = LabelBinarizer()
objetivo = coches['acceptability']
objetivo = codificador_objetivo.fit_transform(objetivo)
objetivo

array([[0, 0, 1, 0],
       [0, 0, 1, 0],
       [0, 0, 1, 0],
       ...,
       [0, 0, 1, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 1]])

In [175]:
from sklearn.model_selection import train_test_split

In [176]:
(atributos_entrenamiento, atributos_prueba,
 objetivo_entrenamiento, objetivo_prueba) = train_test_split(
    atributos, objetivo,
    test_size=.2
)

In [177]:
from keras import Sequential, Input
from keras.layers import Dense

¿Qué es posible obtener con softmax?

![Softmax](https://media.geeksforgeeks.org/wp-content/uploads/20240706012340/Softmax-Activation-Function.webp)

In [178]:
red_coches = Sequential()
red_coches.add(Input(shape=(21,)))
red_coches.add(Dense(16, activation='tanh'))
red_coches.add(Dense(4, activation='softmax'))

In [179]:
red_coches.compile(optimizer='SGD', loss='categorical_crossentropy',
                   metrics=['categorical_accuracy'])

In [180]:
red_coches.fit(atributos_entrenamiento, objetivo_entrenamiento,
               batch_size=256, epochs=50)

Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - categorical_accuracy: 0.4472 - loss: 1.2451  
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - categorical_accuracy: 0.4797 - loss: 1.2041 
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - categorical_accuracy: 0.5159 - loss: 1.1677 
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - categorical_accuracy: 0.5391 - loss: 1.1354 
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - categorical_accuracy: 0.5651 - loss: 1.1066 
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - categorical_accuracy: 0.5803 - loss: 1.0810 
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - categorical_accuracy: 0.6042 - loss: 1.0581 
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - categorical_accuracy: 0.6187 - loss: 1.0376 
Epoch 9/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - categorical_accuracy: 0.6295 - loss: 1.0192 
Epoch 10/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - categorical_accuracy: 0.6375 - loss: 1.0025 
Epoch 11/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

In [181]:
red_coches.evaluate(atributos_prueba, objetivo_prueba)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - categorical_accuracy: 0.7370 - loss: 0.7057  


[0.7057406306266785, 0.736994206905365]

In [182]:
red_coches = Sequential()
red_coches.add(Input(shape=(21,)))
red_coches.add(Dense(8, activation='relu'))
red_coches.add(Dense(16, activation='relu'))
red_coches.add(Dense(32, activation='relu'))
red_coches.add(Dense(4, activation='softmax'))

In [183]:
red_coches.compile(optimizer='SGD', loss='categorical_crossentropy',
                   metrics=['categorical_accuracy'])

In [184]:
red_coches.fit(atributos_entrenamiento, objetivo_entrenamiento,
               batch_size=32, epochs=50, verbose=False)

In [185]:
red_coches.evaluate(atributos_prueba, objetivo_prueba)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - categorical_accuracy: 0.9133 - loss: 0.2122  


[0.21219833195209503, 0.913294792175293]

### Ejercicio 3

Los púlsares son un tipo raro de estrella de neutrones que produce emisiones de radio detectables aquí en la Tierra. Son de considerable interés científico como sondas del espacio-tiempo, el medio interestelar y los estados de la materia.

A medida que los púlsares giran, su haz de emisión recorre el cielo y, cuando cruza nuestra línea de visión, produce un patrón detectable de emisión de radio de banda ancha. Como los púlsares giran rápidamente, este patrón se repite periódicamente. Por tanto, la búsqueda de púlsares implica buscar señales de radio periódicas con grandes radiotelescopios.

Cada púlsar produce un patrón de emisión algo diferente, que varía levemente con cada rotación. Por lo tanto, una detección de señal potencial conocida como «candidata» se promedia a lo largo de muchas rotaciones del púlsar, según lo determinado por la duración de una observación. A falta de información adicional, cada candidato podría describir un púlsar real. Sin embargo, en la práctica, casi todas las detecciones son causadas por interferencias de radiofrecuencia (RFI) y ruido, lo que dificulta encontrar señales legítimas.

El fichero `pulsar_stars.csv` contiene datos acerca de una serie de púlsares reales y de ejemplos espurios producidos por RFI y ruido. Cada candidato se describe mediante ocho atributos continuos extraídos de las señales recibidas.

Se pide construir una red neuronal que permita abordar, con el mayor rendimiento posible, la tarea de determinar si un candidato es o no un púlsar.

### Ejercicio 4

Los [abulones](https://es.wikipedia.org/wiki/Haliotis) son una familia de moluscos gasterópodos. La edad de cada individuo está correlacionada con el número de anillos de su concha y, por tanto, puede determinarse cortando la concha a través del cono, tiñéndola y contando el número de anillos a través de un microscopio. Este procedimiento requiere mucho tiempo y es propenso a errores, por lo que sería preferible poder determinar la edad directamente a partir de medidas físicas más fáciles de obtener.

El fichero `abalone.csv` contiene la siguiente información de distintos individuos de abulones:

* Sexo (`Sex`): atributo discreto con posibles valores `M` (macho), `F` (hembra) e `I` (infante).
* Longitud (`Length`) en milímetros.
* Diámetro (`Diameter`) en milímetros.
* Altura (`Height`) en milímetros.
* Peso total (`Whole_weight`) en gramos.
* Peso sin la concha (`Shucked_weight`) en gramos.
* Peso intestinal (`Viscera_weight`) en gramos.
* Peso de la concha (`Shell_weight`) en gramos.

Se pide construir una red neuronal que permita abordar, con el mayor rendimiento posible, la tarea de predecir el número de anillos (`Rings`) a partir de los atributos anteriores (entonces bastaría sumar 1.5 a ese número de anillos para obtener la edad, en años, del individuo).